In [11]:
pip install roboflow ultralytics pycocotools opencv-python matplotlib seaborn pandas numpy Pillow tqdm scikit-learn pyyaml

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.8/302.8 kB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 58.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 119.6 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 4.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


# **Library**

In [ ]:
import os
import json
import random
import zipfile
import argparse
from collections import Counter

import yaml
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from roboflow import Roboflow


# **Cofiguration Setup Class**

In [ ]:
class CONFIG:
    ROBOFLOW_API_KEY = ""
    ROBOFLOW_WORKSPACE = ""    
    ROBOFLOW_PROJECT = ""     
    ROBOFLOW_VERSION = 41                   
    
    ROBOFLOW_FORMAT_YOLO = "yolov8"
    ROBOFLOW_FORMAT_COCO = "coco-segmentation"

    BASE_DIR = os.path.dirname(os.path.abspath("/kaggle/working/dataset/dataset")) if "__file__" in globals() else os.getcwd()
    DATA_DIR = os.path.join(BASE_DIR, "dataset")            # YOLO-format dataset
    COCO_DATA_DIR = os.path.join(BASE_DIR, "dataset_coco")  # COCO-format dataset
    RUNS_DIR = os.path.join(BASE_DIR, "runs")
    EXPORT_DIR = os.path.join(BASE_DIR, "exported_model")
    EDA_DIR = os.path.join(BASE_DIR, "eda_outputs")
    TEST_PRED_DIR = os.path.join(BASE_DIR, "test_predictions")
    TEST_IMAGES_SUBDIR = "test/images"

    # yolov8n-seg.pt (fastest) ... yolov8s / m / l / x-seg.pt (most accurate)
    BASE_MODEL = "yolov8s-seg.pt"
    EPOCHS = 
    IMG_SIZE = 
    BATCH_SIZE = 
    PATIENCE =             # early stopping patience (epochs w/o improvement)
    DEVICE = 0               # 0 = first GPU, "cpu" = CPU, "0,1" = multi-GPU
    WORKERS = 8
    OPTIMIZER = "Adam"       # "SGD" / "Adam" / "AdamW" / "auto"
    LR0 = 
    SEED = 42
    FREEZE_LAYERS =        # freeze first N backbone layers; 0 = fine-tune all
    RUN_NAME = "instance_seg_finetune"

    CONF_THRESHOLD = 0.25
    IOU_THRESHOLD = 0.45



# **Download Dataset**

    Find the folder that directly contains data.yaml + train/valid/test,
    even if it's nested one or more levels under CONFIG.DATA_DIR.

    Roboflow's download left an unextracted zip sitting in dir_path
    (seen on some Kaggle setups), extract it automatically.

In [ ]:
def resolve_yolo_root():
    direct = os.path.join(CONFIG.COCO_DATA_DIR, "data.yaml")
    if os.path.exists(direct):
        return CONFIG.COCO_DATA_DIR
    if os.path.isdir(CONFIG.COCO_DATA_DIR):
        for root, _dirs, files in os.walk(CONFIG.COCO_DATA_DIR):
            if "data.yaml" in files:
                return root
    print(f"[warn] could not find data.yaml under {CONFIG.COCO_DATA_DIR}; "
          f"using {CONFIG.COCO_DATA_DIR} as-is (things will likely fail)")
    return CONFIG.COCO_DATA_DIR

In [ ]:
def ensure_extracted(dir_path):
    if not os.path.isdir(dir_path):
        return
    for fname in os.listdir(dir_path):
        if fname.lower().endswith(".zip"):
            zip_path = os.path.join(dir_path, fname)
            print(f"[fix] found unextracted zip {zip_path}, extracting...")
            with zipfile.ZipFile(zip_path, "r") as z:
                z.extractall(dir_path)
            print(f"[fix] extracted -> {dir_path}")

In [ ]:
def stage_download():

      rf = Roboflow(api_key=CONFIG.ROBOFLOW_API_KEY)
      project = rf.workspace(CONFIG.ROBOFLOW_WORKSPACE).project(CONFIG.ROBOFLOW_PROJECT)
      version = project.version(CONFIG.ROBOFLOW_VERSION)

      # print(f"Downloading '{CONFIG.ROBOFLOW_PROJECT}' v{CONFIG.ROBOFLOW_VERSION} "
            # f"in YOLOv8-seg format -> {CONFIG.DATA_DIR}")
      # version.download(CONFIG.ROBOFLOW_FORMAT_YOLO, location=CONFIG.DATA_DIR)
      # ensure_extracted(CONFIG.DATA_DIR)

      print(f"Downloading '{CONFIG.ROBOFLOW_PROJECT}' v{CONFIG.ROBOFLOW_VERSION} "
            f"in COCO-segmentation format -> {CONFIG.COCO_DATA_DIR}")
      version.download(CONFIG.ROBOFLOW_FORMAT_COCO, location=CONFIG.COCO_DATA_DIR)
      zip_ensure = "/kaggle/working/dataset_coco/roboflow.zip"
      ensure_extracted(zip_ensure)

      print("\nDataset structure:")
      # print(f"  YOLO format : {CONFIG.DATA_DIR}/{{train,valid,test}}/{{images,labels}}, data.yaml")
      print(f"  COCO format : {CONFIG.COCO_DATA_DIR}/{{train,valid,test}}/_annotations.coco.json")

# **Analysis & EDA**

    Locate the Data annotation json for a split even if Roboflow nested it under an extra folder
    Returns (json_path, image_dir) or (None, None) if not found.

In [ ]:
def find_annotation_file(split):
    split_aliases = {
        "train": ["train"],
        "valid": ["valid", "val", "validation"],
        "test": ["test"],
    }.get(split, [split])

    # exact expected location
    fast_path = os.path.join(CONFIG.COCO_DATA_DIR, split, "_annotations.coco.json")
    if os.path.exists(fast_path):
        return fast_path, os.path.dirname(fast_path)

    # whose parent folder name matches this split
    if os.path.isdir(CONFIG.COCO_DATA_DIR):
        for root, _dirs, files in os.walk(CONFIG.COCO_DATA_DIR):
            parent_name = os.path.basename(root).lower()
            if parent_name not in split_aliases:
                continue
            for fname in files:
                if fname.endswith(".coco.json") or fname == "annotations.json":
                    return os.path.join(root, fname), root

    # any dataset json anywhere with the split name in its path
    if os.path.isdir(CONFIG.COCO_DATA_DIR):
        for root, _dirs, files in os.walk(CONFIG.COCO_DATA_DIR):
            for fname in files:
                if (fname.endswith(".coco.json") or fname == "annotations.json") and \
                        any(alias in root.lower() for alias in split_aliases):
                    return os.path.join(root, fname), root

    return None, None

In [ ]:
SPLITS = ["train", "valid", "test"]




def load_splits():
    from pycocotools.coco import COCO
    cocos = {}
    for split in SPLITS:
        ann_path, _img_dir = _find_annotation_file(split)
        if ann_path:
            print(f"[ok] {split}: found annotations at {ann_path}")
            cocos[split] = COCO(ann_path)
        else:
            print(f"[warn] missing annotations for split '{split}' "
                  f"(searched under {CONFIG.COCO_DATA_DIR})")
    if not cocos:
        print(f"\n[debug] contents of {CONFIG.COCO_DATA_DIR}:")
        if os.path.isdir(CONFIG.COCO_DATA_DIR):
            for root, dirs, files in os.walk(CONFIG.COCO_DATA_DIR):
                depth = root.replace(CONFIG.COCO_DATA_DIR, "").count(os.sep)
                if depth > 2:
                    continue
                indent = "  " * depth
                print(f"{indent}{os.path.basename(root)}/")
                for f in files[:10]:
                    print(f"{indent}  {f}")
        else:
            print("  (directory does not exist)")
    return cocos


def _coco_area_bucket(area):
    if area < 32 ** 2:
        return "small"
    elif area < 96 ** 2:
        return "medium"
    return "large"


def _eda_split_sizes(cocos):
    rows = []
    for split, coco in cocos.items():
        rows.append({
            "split": split,
            "images": len(coco.getImgIds()),
            "annotations": len(coco.getAnnIds()),
            "categories": len(coco.getCatIds()),
        })
    df = pd.DataFrame(rows)
    print("\n=== Split sizes ===")
    print(df.to_string(index=False))
    df.to_csv(os.path.join(CONFIG.EDA_DIR, "split_sizes.csv"), index=False)
    return df


def _eda_class_distribution(cocos):
    records = []
    for split, coco in cocos.items():
        cats = {c["id"]: c["name"] for c in coco.loadCats(coco.getCatIds())}
        for ann in coco.loadAnns(coco.getAnnIds()):
            records.append({
                "split": split,
                "class": cats[ann["category_id"]],
                "area": ann["area"],
                "image_id": ann["image_id"],
            })
    df = pd.DataFrame(records)
    if df.empty:
        print("[warn] no annotations found across splits")
        return df

    counts = df.groupby(["split", "class"]).size().reset_index(name="instance_count")
    plt.figure(figsize=(10, 6))
    sns.barplot(data=counts, x="class", y="instance_count", hue="split")
    plt.xticks(rotation=45, ha="right")
    plt.title("Instance count per class per split")
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG.EDA_DIR, "class_distribution.png"), dpi=150)
    plt.close()

    df["size_bucket"] = df["area"].apply(_coco_area_bucket)
    size_counts = df.groupby(["split", "size_bucket"]).size().reset_index(name="count")
    plt.figure(figsize=(8, 5))
    sns.barplot(data=size_counts, x="size_bucket", y="count", hue="split",
                order=["small", "medium", "large"])
    plt.title("Object size distribution (COCO small/medium/large)")
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG.EDA_DIR, "object_size_distribution.png"), dpi=150)
    plt.close()

    print("\n=== Instance counts per class ===")
    print(counts.pivot(index="class", columns="split", values="instance_count").fillna(0))
    counts.to_csv(os.path.join(CONFIG.EDA_DIR, "class_counts.csv"), index=False)
    return df


def _eda_instances_per_image(cocos, split="train"):
    if split not in cocos:
        return
    coco = cocos[split]
    img_ids = coco.getImgIds()
    counts = [len(coco.getAnnIds(imgIds=i)) for i in img_ids]
    plt.figure(figsize=(8, 5))
    sns.histplot(counts, bins=30)
    plt.title(f"Instances per image ({split} split)")
    plt.xlabel("number of instances")
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG.EDA_DIR, f"instances_per_image_{split}.png"), dpi=150)
    plt.close()
    print(f"\n{split}: mean instances/image = {np.mean(counts):.2f}, "
          f"max = {np.max(counts)}, images with 0 instances = {sum(c == 0 for c in counts)}")


def _eda_visualize_samples(cocos, split="train", n=6):
    from pycocotools import mask as maskUtils
    if split not in cocos:
        return
    coco = cocos[split]
    _ann_path, img_dir = _find_annotation_file(split)
    if img_dir is None:
        img_dir = os.path.join(CONFIG.COCO_DATA_DIR, split)
    img_ids = coco.getImgIds()
    sample_ids = random.sample(img_ids, min(n, len(img_ids)))

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    for ax, img_id in zip(axes, sample_ids):
        img_info = coco.loadImgs(img_id)[0]
        img_path = os.path.join(img_dir, img_info["file_name"])
        img = cv2.imread(img_path)
        if img is None:
            ax.set_title("missing image")
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        ann_ids = coco.getAnnIds(imgIds=img_id)
        anns = coco.loadAnns(ann_ids)
        overlay = img.copy()
        for ann in anns:
            if "segmentation" in ann and ann["segmentation"]:
                if isinstance(ann["segmentation"], list):
                    rles = maskUtils.frPyObjects(ann["segmentation"], img_info["height"], img_info["width"])
                    rle = maskUtils.merge(rles)
                else:
                    rle = ann["segmentation"]
                m = maskUtils.decode(rle)
                color = np.random.randint(0, 255, 3)
                overlay[m == 1] = overlay[m == 1] * 0.5 + color * 0.5
            x, y, w, h = ann["bbox"]
            cv2.rectangle(overlay, (int(x), int(y)), (int(x + w), int(y + h)), (255, 0, 0), 2)

        ax.imshow(overlay.astype(np.uint8))
        ax.set_title(f"img {img_id} ({len(anns)} instances)")
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG.EDA_DIR, f"sample_annotations_{split}.png"), dpi=150)
    plt.close()
    print(f"Saved sample annotation visualization -> {CONFIG.EDA_DIR}/sample_annotations_{split}.png")


def stage_analyze():
    os.makedirs(CONFIG.EDA_DIR, exist_ok=True)
    cocos = _load_coco_splits()
    if not cocos:
        print("No COCO-format splits found. Run the 'download' stage first.")
        return

    _eda_split_sizes(cocos)
    _eda_class_distribution(cocos)
    for split in cocos:
        _eda_instances_per_image(cocos, split)
    _eda_visualize_samples(cocos, split="train")

    print(f"\nAll EDA artifacts saved to: {CONFIG.EDA_DIR}")


# ==========================================================================
# STAGE 3: PREPROCESS / VALIDATE  (uses YOLO-format export)
# ==========================================================================

def _load_data_yaml():
    yolo_root = _resolve_yolo_root()
    yaml_path = os.path.join(yolo_root, "data.yaml")
    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)
    print(f"=== data.yaml (root: {yolo_root}) ===")
    print(data)
    return data, yolo_root


def _validate_split(yolo_root, split, fix=False):
    img_dir = os.path.join(yolo_root, split, "images")
    lbl_dir = os.path.join(yolo_root, split, "labels")
    if not os.path.isdir(img_dir):
        print(f"[warn] split '{split}' not found at {img_dir}")
        return

    images = sorted(f for f in os.listdir(img_dir) if f.lower().endswith((".jpg", ".jpeg", ".png")))
    class_counter = Counter()
    n_missing_label = 0
    n_empty_label = 0
    n_bad_polygon = 0
    n_unreadable = 0

    for img_name in images:
        img_path = os.path.join(img_dir, img_name)
        lbl_path = os.path.join(lbl_dir, os.path.splitext(img_name)[0] + ".txt")

        img = cv2.imread(img_path)
        if img is None:
            n_unreadable += 1
            print(f"[bad image] {img_path}")
            if fix:
                os.remove(img_path)
                if os.path.exists(lbl_path):
                    os.remove(lbl_path)
            continue

        if not os.path.exists(lbl_path):
            n_missing_label += 1
            print(f"[missing label] {lbl_path}")
            continue

        with open(lbl_path, "r") as f:
            lines = [ln.strip() for ln in f if ln.strip()]
        if not lines:
            n_empty_label += 1
            continue

        for ln in lines:
            parts = ln.split()
            cls_id = int(parts[0])
            coords = parts[1:]
            class_counter[cls_id] += 1
            if len(coords) % 2 != 0 or len(coords) < 6:
                n_bad_polygon += 1
                print(f"[bad polygon] {lbl_path}: {ln[:60]}...")
                continue
            vals = list(map(float, coords))
            if any(v < 0.0 or v > 1.0 for v in vals):
                n_bad_polygon += 1
                print(f"[out-of-range coords] {lbl_path}")

    print(f"\n--- {split} ---")
    print(f"images: {len(images)}")
    print(f"unreadable images: {n_unreadable}")
    print(f"missing labels: {n_missing_label}")
    print(f"empty labels: {n_empty_label}")
    print(f"malformed polygons: {n_bad_polygon}")
    print(f"instances per class: {dict(class_counter)}")

    if class_counter:
        max_c = max(class_counter.values())
        min_c = min(class_counter.values())
        if min_c > 0 and max_c / min_c > 5:
            print(f"[note] class imbalance detected (max/min ratio = {max_c/min_c:.1f}). "
                  f"Consider class-weighted loss or oversampling rare classes.")


def stage_preprocess(fix=False):
    _data, yolo_root = _load_data_yaml()
    for split in ["train", "val", "valid", "test"]:
        split_dir = os.path.join(yolo_root, split)
        if os.path.isdir(split_dir):
            _validate_split(yolo_root, split, fix=fix)
    print("\nValidation complete.")


# ==========================================================================
# STAGE 4: TRAIN / FINE-TUNE  (Ultralytics YOLOv8-seg)
# ==========================================================================

def _patch_data_yaml_paths():
    """Rewrite train/val/test paths in data.yaml as absolute paths, resolving
    the real dataset root even if Roboflow nested it under an extra folder."""
    yolo_root = _resolve_yolo_root()
    yaml_path = os.path.join(yolo_root, "data.yaml")
    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)

    for split_key, subdir in [("train", "train/images"), ("val", "valid/images"), ("test", "test/images")]:
        candidate = os.path.join(yolo_root, subdir)
        if os.path.isdir(candidate):
            data[split_key] = candidate

    with open(yaml_path, "w") as f:
        yaml.safe_dump(data, f)

    print(f"Patched data.yaml paths (root: {yolo_root}):\n{data}")
    return yaml_path


def stage_train():
    from ultralytics import YOLO

    data_yaml = _patch_data_yaml_paths()
    model = YOLO(CONFIG.BASE_MODEL)  # loads pretrained COCO weights

    model.train(
        data=data_yaml,
        epochs=CONFIG.EPOCHS,
        imgsz=CONFIG.IMG_SIZE,
        batch=CONFIG.BATCH_SIZE,
        device=CONFIG.DEVICE,
        workers=CONFIG.WORKERS,
        optimizer=CONFIG.OPTIMIZER,
        lr0=CONFIG.LR0,
        patience=CONFIG.PATIENCE,
        seed=CONFIG.SEED,
        freeze=CONFIG.FREEZE_LAYERS,
        project=CONFIG.RUNS_DIR,
        name=CONFIG.RUN_NAME,
        # augmentation (tune as needed for your domain)
        degrees=10.0,
        translate=0.1,
        scale=0.5,
        shear=2.0,
        flipud=0.0,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.1,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        val=True,
        plots=True,
        save=True,
        exist_ok=True,
    )

    best_weights = os.path.join(CONFIG.RUNS_DIR, CONFIG.RUN_NAME, "weights", "best.pt")
    print(f"\nTraining complete. Best weights saved at:\n  {best_weights}")
    return best_weights


# ==========================================================================
# STAGE 5: EVALUATE  (Ultralytics val() + independent pycocotools COCOeval)
# ==========================================================================

def _default_weights_path():
    return os.path.join(CONFIG.RUNS_DIR, CONFIG.RUN_NAME, "weights", "best.pt")


def _ultralytics_val(weights_path, split="val"):
    from ultralytics import YOLO

    print(f"\n=== Ultralytics built-in validation ({split}) ===")
    model = YOLO(weights_path)
    yolo_root = _resolve_yolo_root()
    metrics = model.val(
        data=os.path.join(yolo_root, "data.yaml"),
        split=split,
        imgsz=CONFIG.IMG_SIZE,
        conf=CONFIG.CONF_THRESHOLD,
        iou=CONFIG.IOU_THRESHOLD,
        plots=True,
        save_json=True,
    )
    print("Box mAP50-95:", metrics.box.map)
    print("Box mAP50   :", metrics.box.map50)
    print("Mask mAP50-95:", metrics.seg.map)
    print("Mask mAP50   :", metrics.seg.map50)
    return metrics


def _predictions_to_coco_json(weights_path, coco_gt, img_dir, out_json):
    from ultralytics import YOLO

    model = YOLO(weights_path)
    results_coco = []
    img_ids = coco_gt.getImgIds()

    for img_id in img_ids:
        img_info = coco_gt.loadImgs(img_id)[0]
        img_path = os.path.join(img_dir, img_info["file_name"])
        if not os.path.exists(img_path):
            continue

        preds = model.predict(
            img_path, conf=CONFIG.CONF_THRESHOLD, iou=CONFIG.IOU_THRESHOLD, verbose=False
        )[0]

        if preds.masks is None:
            continue

        cat_id_map = {i: c["id"] for i, c in enumerate(coco_gt.loadCats(coco_gt.getCatIds()))}

        for box, score, cls, mask_poly in zip(
            preds.boxes.xywh.cpu().numpy(),
            preds.boxes.conf.cpu().numpy(),
            preds.boxes.cls.cpu().numpy().astype(int),
            preds.masks.xy,
        ):
            x_c, y_c, w, h = box
            x = x_c - w / 2
            y = y_c - h / 2
            segmentation = mask_poly.flatten().tolist()
            results_coco.append({
                "image_id": img_id,
                "category_id": cat_id_map.get(cls, cls),
                "bbox": [float(x), float(y), float(w), float(h)],
                "score": float(score),
                "segmentation": [segmentation],
            })

    with open(out_json, "w") as f:
        json.dump(results_coco, f)
    print(f"Saved {len(results_coco)} predictions -> {out_json}")
    return out_json


def _coco_style_eval(weights_path, split="test"):
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

    print(f"\n=== Independent COCOeval ({split}) ===")
    ann_path, img_dir = _find_annotation_file(split)
    if not ann_path:
        print(f"[warn] no COCO annotations found for split '{split}' "
              f"under {CONFIG.COCO_DATA_DIR}, skipping")
        return

    coco_gt = COCO(ann_path)
    pred_json = os.path.join(CONFIG.BASE_DIR, f"predictions_{split}.json")
    _predictions_to_coco_json(weights_path, coco_gt, img_dir, pred_json)

    coco_dt = coco_gt.loadRes(pred_json)

    for iou_type in ["bbox", "segm"]:
        print(f"\n--- {iou_type.upper()} metrics ---")
        coco_eval = COCOeval(coco_gt, coco_dt, iou_type)
        coco_eval.evaluate()
        coco_eval.accumulate()
        coco_eval.summarize()


def stage_evaluate(weights_path=None):
    weights_path = weights_path or _default_weights_path()
    _ultralytics_val(weights_path, split="val")
    _coco_style_eval(weights_path, split="test")


# ==========================================================================
# STAGE 6: TEST / VISUALIZE PREDICTIONS
# ==========================================================================

def stage_test(weights_path=None, n_images=12):
    from ultralytics import YOLO

    weights_path = weights_path or _default_weights_path()
    yolo_root = _resolve_yolo_root()
    test_dir = os.path.join(yolo_root, CONFIG.TEST_IMAGES_SUBDIR)
    os.makedirs(CONFIG.TEST_PRED_DIR, exist_ok=True)

    model = YOLO(weights_path)

    all_images = [f for f in os.listdir(test_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    sample = random.sample(all_images, min(n_images, len(all_images)))

    for img_name in sample:
        img_path = os.path.join(test_dir, img_name)
        results = model.predict(
            img_path, conf=CONFIG.CONF_THRESHOLD, iou=CONFIG.IOU_THRESHOLD, verbose=False,
        )
        annotated = results[0].plot()  # numpy BGR image with boxes+masks+labels drawn
        out_path = os.path.join(CONFIG.TEST_PRED_DIR, f"pred_{img_name}")
        cv2.imwrite(out_path, annotated)
        print(f"Saved: {out_path}  ({len(results[0].boxes)} detections)")

    print(f"\nAll annotated test predictions saved to: {CONFIG.TEST_PRED_DIR}")


# ==========================================================================
# STAGE 7: EXPORT MODEL
# ==========================================================================

def stage_export(weights_path=None, fmt="onnx"):
    import shutil
    from ultralytics import YOLO

    weights_path = weights_path or _default_weights_path()
    os.makedirs(CONFIG.EXPORT_DIR, exist_ok=True)
    model = YOLO(weights_path)
    exported_path = model.export(format=fmt, imgsz=CONFIG.IMG_SIZE)
    dest = os.path.join(CONFIG.EXPORT_DIR, os.path.basename(exported_path))
    shutil.copy(exported_path, dest)
    print(f"Exported model ({fmt}) -> {dest}")
    return dest


# ==========================================================================
# CLI ENTRYPOINT
# ==========================================================================


In [13]:
stage_download()
stage_analyze()
stage_preprocess()

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /kaggle/working/dataset in yolov8:: 100%|██████████| 246134/246134 [00:30<00:00, 8133.51it/s]



Extracting Dataset Version Zip to /kaggle/working/dataset_coco in coco-segmentation:: 100%|██████████| 123069/123069 [00:16<00:00, 7280.73it/s]



Done. Dataset structure:
  YOLO format : /kaggle/working/dataset/{train,valid,test}/{images,labels}, data.yaml
  COCO format : /kaggle/working/dataset_coco/{train,valid,test}/_annotations.coco.json
  (note: Roboflow may nest these one folder deeper under a project-name folder -- every stage below auto-detects that, no action needed)
[ok] train: found annotations at /kaggle/working/dataset_coco/train/_annotations.coco.json
loading annotations into memory...
Done (t=9.60s)
creating index...
index created!
[ok] valid: found annotations at /kaggle/working/dataset_coco/valid/_annotations.coco.json
loading annotations into memory...
Done (t=2.04s)
creating index...
index created!
[ok] test: found annotations at /kaggle/working/dataset_coco/test/_annotations.coco.json
loading annotations into memory...
Done (t=1.26s)
creating index...
index created!

=== Split sizes ===
split  images  annotations  categories
train   90275       660391          81
valid   20989       152194          81
 test 

In [ ]:
best = stage_train()


Patched data.yaml paths (root: /kaggle/working/dataset):
{'names': ['airplane', 'apple', 'backpack', 'banana', 'baseball bat', 'baseball glove', 'bear', 'bed', 'bench', 'bicycle', 'bird', 'boat', 'book', 'bottle', 'bowl', 'broccoli', 'bus', 'cake', 'car', 'carrot', 'cat', 'cell phone', 'chair', 'clock', 'couch', 'cow', 'cup', 'dining table', 'dog', 'donut', 'elephant', 'fire hydrant', 'fork', 'frisbee', 'giraffe', 'hair drier', 'handbag', 'horse', 'hot dog', 'keyboard', 'kite', 'knife', 'laptop', 'microwave', 'motorcycle', 'mouse', 'orange', 'oven', 'parking meter', 'person', 'pizza', 'potted plant', 'refrigerator', 'remote', 'sandwich', 'scissors', 'sheep', 'sink', 'skateboard', 'skis', 'snowboard', 'spoon', 'sports ball', 'stop sign', 'suitcase', 'surfboard', 'teddy bear', 'tennis racket', 'tie', 'toaster', 'toilet', 'toothbrush', 'traffic light', 'train', 'truck', 'tv', 'umbrella', 'vase', 'wine glass', 'zebra'], 'nc': 80, 'roboflow': {'license': 'Private', 'project': 'coco-dataset-

`get_trial_id` is meant to only be called inside a function that is executed by a Tuner or Trainer. Returning `None`.


      2/100      3.85G      1.333       3.06      1.831      1.357          0         30        640: 100% ━━━━━━━━━━━━ 5643/5643 3.4it/s 27:460.6ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 656/656 2.1it/s 5:150.5ss
                   all      20989     152194      0.597      0.436      0.461      0.307      0.593      0.416      0.435       0.26

      Epoch    GPU_mem   box_loss   seg_loss   cls_loss   dfl_loss   sem_loss  Instances       Size
      3/100      3.85G      1.365        3.1       1.88      1.385          0         22        640: 100% ━━━━━━━━━━━━ 5643/5643 3.5it/s 27:150.2ss
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 656/656 2.1it/s 5:160.5ss
                   all      20989     152194      0.605      0.435      0.467      0.312      0.604      0.415    

In [14]:
stage_evaluate(weights_path="/kaggle/working/yolov8s-seg.pt")
stage_test(weights_path="/kaggle/working/yolov8s-seg.pt")
stage_export(weights_path="/kaggle/working/yolov8s-seg.pt")


=== Ultralytics built-in validation (val) ===
Ultralytics 8.4.142 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
YOLOv8s-seg summary (fused): 85 layers, 11,810,560 parameters, 0 gradients, 40.1 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 848.1±295.9 MB/s, size: 20.7 KB)
val: Scanning /kaggle/working/dataset/valid/labels... 20989 images, 206 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 20989/20989 936.4it/s 22.4s0.1ss
val: New cache created: /kaggle/working/dataset/valid/labels.cache
requirements: Ultralytics requirement ['faster-coco-eval>=1.6.7'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 2 packages in 200ms
Prepared 1 package in 30ms
Installed 1 package in 5ms
 + faster-coco-eval==1.8.0

requirements: AutoUpdate success ✅ 0.6s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect

                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95)     Mask(P    

'/kaggle/working/exported_model/yolov8s-seg.onnx'

# **Libraries**

In [83]:
import os
import json
import random
import argparse
from collections import Counter

import yaml
import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# **Downloading**

In [84]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="bbrsCDbFqgCpWkPYiqdt")
project = rf.workspace("microsoft").project("coco-dataset-vdnr1")
version = project.version(41)
dataset = version.download("coco-segmentation")

loading Roboflow workspace...
loading Roboflow project...


# **SetUp**

In [103]:
class CONFIG:

    ROBOFLOW_API_KEY = os.environ.get("bbrsCDbFqgCpWkPYiqdt", "rf_qEveYiE2oLgHt46wAKh3m5nKRIi2")
    ROBOFLOW_WORKSPACE = "microsoft"    
    ROBOFLOW_PROJECT = "coco-dataset-vdnr1"     
    ROBOFLOW_VERSION = 41                   
    
    # ROBOFLOW_FORMAT_YOLO = "yolov8"
    ROBOFLOW_FORMAT_COCO = "coco-segmentation"

    # Paths
    BASE_DIR = os.path.dirname(os.path.abspath("/kaggle/working/COCO-Dataset-41"))
    DATA_DIR = os.path.join(BASE_DIR, "COCO-Dataset-41")            # YOLO-format dataset
    COCO_DATA_DIR = os.path.join(BASE_DIR, "dataset_coco")  # COCO-format dataset
    # RUNS_DIR = os.path.join(BASE_DIR, "runs")
    # EXPORT_DIR = os.path.join(BASE_DIR, "exported_model")
    EDA_DIR = os.path.join(BASE_DIR, "eda_outputs")
    TEST_PRED_DIR = os.path.join(BASE_DIR, "test_predictions")
    TEST_IMAGES_SUBDIR = "test/images"

    # Model / training
    # yolov8n-seg.pt (fastest) ... yolov8s / m / l / x-seg.pt (most accurate)
    BASE_MODEL = "yolov8s-seg.pt"
    EPOCHS = 100
    IMG_SIZE = 640
    BATCH_SIZE = 16
    PATIENCE = 20            # early stopping patience (epochs w/o improvement)
    DEVICE = 0               # 0 = first GPU, "cpu" = CPU, "0,1" = multi-GPU
    WORKERS = 8
    OPTIMIZER = "auto"       # "SGD" / "Adam" / "AdamW" / "auto"
    LR0 = 0.01
    SEED = 42
    FREEZE_LAYERS = 10       # freeze first N backbone layers; 0 = fine-tune all
    RUN_NAME = "instance_seg_finetune"

    # Evaluation / inference
    CONF_THRESHOLD = 0.25
    IOU_THRESHOLD = 0.45

In [87]:
print(f"Downloading '{CONFIG.ROBOFLOW_PROJECT}' v{CONFIG.ROBOFLOW_VERSION} "
      f"in COCO-segmentation format -> {CONFIG.DATA_DIR}")

print("\nDataset structure:")
print(f"  COCO format : {CONFIG.DATA_DIR}/{{train,valid,test}}/_annotations.coco.json")
print(f"  COCO format : {CONFIG.COCO_DATA_DIR}/{{train,valid,test}}/_annotations.coco.json")


Dataset structure:
  COCO format : /kaggle/working/COCO-Dataset-41/{train,valid,test}/_annotations.coco.json
  COCO format : /kaggle/working/dataset_coco/{train,valid,test}/_annotations.coco.json


# **Split & EDA Implementation**

## **Split Function**

In [95]:
def find_annotation_file(split):
    """Locate the COCO annotation json for a split even if Roboflow nested
    it under an extra folder, or used 'val' instead of 'valid', etc.
    Returns (json_path, image_dir) or (None, None) if not found."""
    split_aliases = {
        "train": ["train"],
        "valid": ["valid"],
        "test": ["test"],
    }.get(split, [split])
 
    # 1) fast path: exact expected location
    fast_path = os.path.join(CONFIG.DATA_DIR, split, "_annotations.coco.json")
    if os.path.exists(fast_path):
        return fast_path, os.path.dirname(fast_path)
 
    # 2) walk the whole COCO_DATA_DIR looking for any *.coco.json (or
    #    annotations.json) whose parent folder name matches this split
    if os.path.isdir(CONFIG.COCO_DATA_DIR):
        for root, _dirs, files in os.walk(CONFIG.COCO_DATA_DIR):
            parent_name = os.path.basename(root).lower()
            if parent_name not in split_aliases:
                continue
            for fname in files:
                if fname.endswith(".coco.json") or fname == "annotations.json":
                    return os.path.join(root, fname), root
 
    # 3) last resort: any coco json anywhere with the split name in its path
    if os.path.isdir(CONFIG.COCO_DATA_DIR):
        for root, _dirs, files in os.walk(CONFIG.COCO_DATA_DIR):
            for fname in files:
                if (fname.endswith(".coco.json") or fname == "annotations.json") and \
                        any(alias in root.lower() for alias in split_aliases):
                    return os.path.join(root, fname), root
 
    return None, None

In [89]:
def load_coco_splits(SPLITS):
    from pycocotools.coco import COCO
    cocos = {}
    for split in SPLITS:
        ann_path, _img_dir = find_annotation_file(split)
        if ann_path:
            print(f"[ok] {split}: found annotations at {ann_path}")
            cocos[split] = COCO(ann_path)
        else:
            print(f"[warn] missing annotations for split '{split}' "
                  f"(searched under {CONFIG.COCO_DATA_DIR})")
    if not cocos:
        print(f"\n[debug] contents of {CONFIG.COCO_DATA_DIR}:")
        if os.path.isdir(CONFIG.COCO_DATA_DIR):
            for root, dirs, files in os.walk(CONFIG.COCO_DATA_DIR):
                depth = root.replace(CONFIG.COCO_DATA_DIR, "").count(os.sep)
                if depth > 2:
                    continue
                indent = "  " * depth
                print(f"{indent}{os.path.basename(root)}/")
                for f in files[:10]:
                    print(f"{indent}  {f}")
        else:
            print("  (directory does not exist)")
    return cocos

## **EDA Functions**

In [97]:
os.makedirs("/kaggle/working/eda_outputs", exist_ok=True)

def eda_split_sizes(cocos):
    rows = []
    for split, coco in cocos.items():
        rows.append({
            "split": split,
            "images": len(coco.getImgIds()),
            "annotations": len(coco.getAnnIds()),
            "categories": len(coco.getCatIds()),
        })
    split_sizes = pd.DataFrame(rows)
    print("\nSplit sizes")
    print(split_sizes.to_string(index=False))
    split_sizes.to_csv(os.path.join(CONFIG.EDA_DIR, "split_sizes.csv"), index=False)
    return split_sizes

In [91]:
def coco_area_bucket(area):
    if area < 32 ** 2:
        return "small"
    elif area < 96 ** 2:
        return "medium"
    return "large"

In [99]:
def eda_class_distribution(cocos):
    records = []
    for split, coco in cocos.items():
        cats = {c["id"]: c["name"] for c in coco.loadCats(coco.getCatIds())}
        for ann in coco.loadAnns(coco.getAnnIds()):
            records.append({
                "split": split,
                "class": cats[ann["category_id"]],
                "area": ann["area"],
                "image_id": ann["image_id"],
            })
    class_counts = pd.DataFrame(records)
    if df.empty:
        print("[warn] no annotations found across splits")
        return class_counts

    counts = df.groupby(["split", "class"]).size().reset_index(name="instance_count")
    plt.figure(figsize=(10, 6))
    sns.barplot(data=counts, x="class", y="instance_count", hue="split")
    plt.xticks(rotation=45, ha="right")
    plt.title("Instance count per class per split")
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG.EDA_DIR, "class_distribution.png"), dpi=150)
    plt.close()

    df["size_bucket"] = df["area"].apply(coco_area_bucket)
    size_counts = df.groupby(["split", "size_bucket"]).size().reset_index(name="count")
    plt.figure(figsize=(8, 5))
    sns.barplot(data=size_counts, x="size_bucket", y="count", hue="split",
                order=["small", "medium", "large"])
    plt.title("Object size distribution (COCO small/medium/large)")
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG.EDA_DIR, "object_size_distribution.png"), dpi=150)
    plt.close()

    print("\n=== Instance counts per class ===")
    print(counts.pivot(index="class", columns="split", values="instance_count").fillna(0))
    counts.to_csv(os.path.join(CONFIG.EDA_DIR, "class_counts.csv"), index=False)
    return class_counts

In [51]:
def eda_instances_per_image(cocos, split="train"):
    if split not in cocos:
        return
    coco = cocos[split]
    img_ids = coco.getImgIds()
    counts = [len(coco.getAnnIds(imgIds=i)) for i in img_ids]
    plt.figure(figsize=(8, 5))
    sns.histplot(counts, bins=30)
    plt.title(f"Instances per image ({split} split)")
    plt.xlabel("number of instances")
    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG.EDA_DIR, f"instances_per_image_{split}.png"), dpi=150)
    plt.close()
    print(f"\n{split}: mean instances/image = {np.mean(counts):.2f}, "
          f"max = {np.max(counts)}, images with 0 instances = {sum(c == 0 for c in counts)}")

In [58]:
def eda_visualize_samples(cocos, split="train", n=6):
    from pycocotools import mask as maskUtils
    if split not in cocos:
        return
    coco = cocos[split]
    img_dir = os.path.join(CONFIG.COCO_DATA_DIR, split)
    img_ids = coco.getImgIds()
    sample_ids = random.sample(img_ids, min(n, len(img_ids)))

    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()

    for ax, img_id in zip(axes, sample_ids):
        img_info = coco.loadImgs(img_id)[0]
        img_path = os.path.join(img_dir, img_info["file_name"])
        img = cv2.imread(img_path)
        if img is None:
            ax.set_title("missing image")
            continue
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

        ann_ids = coco.getAnnIds(imgIds=img_id)
        anns = coco.loadAnns(ann_ids)
        overlay = img.copy()
        for ann in anns:
            if "segmentation" in ann and ann["segmentation"]:
                if isinstance(ann["segmentation"], list):
                    rles = maskUtils.frPyObjects(ann["segmentation"], img_info["height"], img_info["width"])
                    rle = maskUtils.merge(rles)
                else:
                    rle = ann["segmentation"]
                m = maskUtils.decode(rle)
                color = np.random.randint(0, 255, 3)
                overlay[m == 1] = overlay[m == 1] * 0.5 + color * 0.5
            x, y, w, h = ann["bbox"]
            cv2.rectangle(overlay, (int(x), int(y)), (int(x + w), int(y + h)), (255, 0, 0), 2)

        ax.imshow(overlay.astype(np.uint8))
        ax.set_title(f"img {img_id} ({len(anns)} instances)")
        ax.axis("off")

    plt.tight_layout()
    plt.savefig(os.path.join(CONFIG.EDA_DIR, f"sample_annotations_{split}.png"), dpi=150)
    plt.close()
    print(f"Saved sample annotation visualization -> {CONFIG.EDA_DIR}/sample_annotations_{split}.png")

## **Run ALL**

In [100]:
SPLITS = ["train", "valid", "test"]

def stage_analyze(SPLITS):
    os.makedirs(CONFIG.EDA_DIR, exist_ok=True)
    cocos = load_coco_splits(SPLITS)
    if not cocos:
        return

    eda_split_sizes(cocos)
    eda_class_distribution(cocos)
    for split in cocos:
        eda_instances_per_image(cocos, split)
    eda_visualize_samples(cocos, split="train")

    print(f"\nAll EDA artifacts saved to: {CONFIG.EDA_DIR}")

stage_analyze(SPLITS)

[ok] train: found annotations at /kaggle/working/COCO-Dataset-41/train/_annotations.coco.json
loading annotations into memory...
Done (t=10.68s)
creating index...


KeyboardInterrupt: 

# **Preprocessing & Validation**

## **Functions**

In [61]:
def load_data_yaml():
    yaml_path = os.path.join(CONFIG.DATA_DIR, "data.yaml")
    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)
    print("=== data.yaml ===")
    print(data)
    return data

In [62]:
def validate_split(split, fix=False):
    img_dir = os.path.join(CONFIG.DATA_DIR, split, "images")
    lbl_dir = os.path.join(CONFIG.DATA_DIR, split, "labels")
    if not os.path.isdir(img_dir):
        print(f"[warn] split '{split}' not found at {img_dir}")
        return

    images = sorted(f for f in os.listdir(img_dir) if f.lower().endswith((".jpg", ".jpeg", ".png")))
    class_counter = Counter()
    n_missing_label = 0
    n_empty_label = 0
    n_bad_polygon = 0
    n_unreadable = 0

    for img_name in images:
        img_path = os.path.join(img_dir, img_name)
        lbl_path = os.path.join(lbl_dir, os.path.splitext(img_name)[0] + ".txt")

        img = cv2.imread(img_path)
        if img is None:
            n_unreadable += 1
            print(f"[bad image] {img_path}")
            if fix:
                os.remove(img_path)
                if os.path.exists(lbl_path):
                    os.remove(lbl_path)
            continue

        if not os.path.exists(lbl_path):
            n_missing_label += 1
            print(f"[missing label] {lbl_path}")
            continue

        with open(lbl_path, "r") as f:
            lines = [ln.strip() for ln in f if ln.strip()]
        if not lines:
            n_empty_label += 1
            continue

        for ln in lines:
            parts = ln.split()
            cls_id = int(parts[0])
            coords = parts[1:]
            class_counter[cls_id] += 1
            if len(coords) % 2 != 0 or len(coords) < 6:
                n_bad_polygon += 1
                print(f"[bad polygon] {lbl_path}: {ln[:60]}...")
                continue
            vals = list(map(float, coords))
            if any(v < 0.0 or v > 1.0 for v in vals):
                n_bad_polygon += 1
                print(f"[out-of-range coords] {lbl_path}")

    print(f"\n--- {split} ---")
    print(f"images: {len(images)}")
    print(f"unreadable images: {n_unreadable}")
    print(f"missing labels: {n_missing_label}")
    print(f"empty labels: {n_empty_label}")
    print(f"malformed polygons: {n_bad_polygon}")
    print(f"instances per class: {dict(class_counter)}")

    if class_counter:
        max_c = max(class_counter.values())
        min_c = min(class_counter.values())
        if min_c > 0 and max_c / min_c > 5:
            print(f"[note] class imbalance detected (max/min ratio = {max_c/min_c:.1f}). "
                  f"Consider class-weighted loss or oversampling rare classes.")

## **Run ALL**

In [67]:
!find /kaggle/working -maxdepth 3 -type d

/kaggle/working
/kaggle/working/eda_outputs
/kaggle/working/.virtual_documents
/kaggle/working/dataset_coco
/kaggle/working/COCO-Dataset-41
/kaggle/working/COCO-Dataset-41/valid
/kaggle/working/COCO-Dataset-41/train
/kaggle/working/COCO-Dataset-41/test


In [ ]:
!find /kaggle/working/COCO-Dataset-41/train -maxdepth 2

/kaggle/working/COCO-Dataset-41/train
/kaggle/working/COCO-Dataset-41/train/000000000025_jpg.rf.efe46dde216acd7ff2b67cac0120c8e3.jpg
/kaggle/working/COCO-Dataset-41/train/000000000030_jpg.rf.6924d2c12040981f985f26e344b17a1a.jpg
/kaggle/working/COCO-Dataset-41/train/000000000034_jpg.rf.bc9d984962bb7304a18e03fee904e6f1.jpg
/kaggle/working/COCO-Dataset-41/train/000000000036_jpg.rf.06baeaa390cc28fff5f1a15ed81fcf11.jpg
/kaggle/working/COCO-Dataset-41/train/000000000042_jpg.rf.c2441c7e9abb8a6af0abe52275035e53.jpg
/kaggle/working/COCO-Dataset-41/train/000000000049_jpg.rf.359c20733510f44941e3ada76d070db4.jpg
/kaggle/working/COCO-Dataset-41/train/000000000061_jpg.rf.071f99025317631682d2133332ccb769.jpg
/kaggle/working/COCO-Dataset-41/train/000000000064_jpg.rf.d3e20a6ab21ae88119ed44eadb4b1980.jpg
/kaggle/working/COCO-Dataset-41/train/000000000071_jpg.rf.9936b9a9b399bb11e217634597628f7d.jpg
/kaggle/working/COCO-Dataset-41/train/000000000072_jpg.rf.4d03deb7e0fcc44c7d205041f8401fa4.jpg
/kaggle/work

In [64]:
def stage_preprocess(fix=False):
    load_data_yaml()
    for split in ["train", "valid", "test"]:
        split_dir = os.path.join(CONFIG.DATA_DIR, split)
        if os.path.isdir(split_dir):
            validate_split(split, fix=fix)

stage_preprocess(fix=False)

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/COCO-Dataset-41/data.yaml'

In [1]:



# ==========================================================================
# STAGE 4: TRAIN / FINE-TUNE  (Ultralytics YOLOv8-seg)
# ==========================================================================

def _patch_data_yaml_paths():
    """Rewrite train/val/test paths in data.yaml as absolute paths."""
    yaml_path = os.path.join(CONFIG.DATA_DIR, "data.yaml")
    with open(yaml_path, "r") as f:
        data = yaml.safe_load(f)

    for split_key, subdir in [("train", "train/images"), ("val", "valid/images"), ("test", "test/images")]:
        candidate = os.path.join(CONFIG.DATA_DIR, subdir)
        if os.path.isdir(candidate):
            data[split_key] = candidate

    with open(yaml_path, "w") as f:
        yaml.safe_dump(data, f)

    print(f"Patched data.yaml paths:\n{data}")
    return yaml_path


def stage_train():
    from ultralytics import YOLO

    data_yaml = _patch_data_yaml_paths()
    model = YOLO(CONFIG.BASE_MODEL)  # loads pretrained COCO weights

    model.train(
        data=data_yaml,
        epochs=CONFIG.EPOCHS,
        imgsz=CONFIG.IMG_SIZE,
        batch=CONFIG.BATCH_SIZE,
        device=CONFIG.DEVICE,
        workers=CONFIG.WORKERS,
        optimizer=CONFIG.OPTIMIZER,
        lr0=CONFIG.LR0,
        patience=CONFIG.PATIENCE,
        seed=CONFIG.SEED,
        freeze=CONFIG.FREEZE_LAYERS,
        project=CONFIG.RUNS_DIR,
        name=CONFIG.RUN_NAME,
        # augmentation (tune as needed for your domain)
        degrees=10.0,
        translate=0.1,
        scale=0.5,
        shear=2.0,
        flipud=0.0,
        fliplr=0.5,
        mosaic=1.0,
        mixup=0.1,
        hsv_h=0.015,
        hsv_s=0.7,
        hsv_v=0.4,
        val=True,
        plots=True,
        save=True,
        exist_ok=True,
    )

    best_weights = os.path.join(CONFIG.RUNS_DIR, CONFIG.RUN_NAME, "weights", "best.pt")
    print(f"\nTraining complete. Best weights saved at:\n  {best_weights}")
    return best_weights


# ==========================================================================
# STAGE 5: EVALUATE  (Ultralytics val() + independent pycocotools COCOeval)
# ==========================================================================

def _default_weights_path():
    return os.path.join(CONFIG.RUNS_DIR, CONFIG.RUN_NAME, "weights", "best.pt")


def _ultralytics_val(weights_path, split="val"):
    from ultralytics import YOLO

    print(f"\n=== Ultralytics built-in validation ({split}) ===")
    model = YOLO(weights_path)
    metrics = model.val(
        data=os.path.join(CONFIG.DATA_DIR, "data.yaml"),
        split=split,
        imgsz=CONFIG.IMG_SIZE,
        conf=CONFIG.CONF_THRESHOLD,
        iou=CONFIG.IOU_THRESHOLD,
        plots=True,
        save_json=True,
    )
    print("Box mAP50-95:", metrics.box.map)
    print("Box mAP50   :", metrics.box.map50)
    print("Mask mAP50-95:", metrics.seg.map)
    print("Mask mAP50   :", metrics.seg.map50)
    return metrics


def _predictions_to_coco_json(weights_path, coco_gt, img_dir, out_json):
    from ultralytics import YOLO

    model = YOLO(weights_path)
    results_coco = []
    img_ids = coco_gt.getImgIds()

    for img_id in img_ids:
        img_info = coco_gt.loadImgs(img_id)[0]
        img_path = os.path.join(img_dir, img_info["file_name"])
        if not os.path.exists(img_path):
            continue

        preds = model.predict(
            img_path, conf=CONFIG.CONF_THRESHOLD, iou=CONFIG.IOU_THRESHOLD, verbose=False
        )[0]

        if preds.masks is None:
            continue

        cat_id_map = {i: c["id"] for i, c in enumerate(coco_gt.loadCats(coco_gt.getCatIds()))}

        for box, score, cls, mask_poly in zip(
            preds.boxes.xywh.cpu().numpy(),
            preds.boxes.conf.cpu().numpy(),
            preds.boxes.cls.cpu().numpy().astype(int),
            preds.masks.xy,
        ):
            x_c, y_c, w, h = box
            x = x_c - w / 2
            y = y_c - h / 2
            segmentation = mask_poly.flatten().tolist()
            results_coco.append({
                "image_id": img_id,
                "category_id": cat_id_map.get(cls, cls),
                "bbox": [float(x), float(y), float(w), float(h)],
                "score": float(score),
                "segmentation": [segmentation],
            })

    with open(out_json, "w") as f:
        json.dump(results_coco, f)
    print(f"Saved {len(results_coco)} predictions -> {out_json}")
    return out_json


def _coco_style_eval(weights_path, split="test"):
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval

    print(f"\n=== Independent COCOeval ({split}) ===")
    ann_path = os.path.join(CONFIG.COCO_DATA_DIR, split, "_annotations.coco.json")
    img_dir = os.path.join(CONFIG.COCO_DATA_DIR, split)
    if not os.path.exists(ann_path):
        print(f"[warn] no COCO annotations at {ann_path}, skipping")
        return

    coco_gt = COCO(ann_path)
    pred_json = os.path.join(CONFIG.BASE_DIR, f"predictions_{split}.json")
    _predictions_to_coco_json(weights_path, coco_gt, img_dir, pred_json)

    coco_dt = coco_gt.loadRes(pred_json)

    for iou_type in ["bbox", "segm"]:
        print(f"\n--- {iou_type.upper()} metrics ---")
        coco_eval = COCOeval(coco_gt, coco_dt, iou_type)
        coco_eval.evaluate()
        coco_eval.accumulate()
        coco_eval.summarize()


def stage_evaluate(weights_path=None):
    weights_path = weights_path or _default_weights_path()
    _ultralytics_val(weights_path, split="val")
    _coco_style_eval(weights_path, split="test")


# ==========================================================================
# STAGE 6: TEST / VISUALIZE PREDICTIONS
# ==========================================================================

def stage_test(weights_path=None, n_images=12):
    from ultralytics import YOLO

    weights_path = weights_path or _default_weights_path()
    test_dir = os.path.join(CONFIG.DATA_DIR, CONFIG.TEST_IMAGES_SUBDIR)
    os.makedirs(CONFIG.TEST_PRED_DIR, exist_ok=True)

    model = YOLO(weights_path)

    all_images = [f for f in os.listdir(test_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    sample = random.sample(all_images, min(n_images, len(all_images)))

    for img_name in sample:
        img_path = os.path.join(test_dir, img_name)
        results = model.predict(
            img_path, conf=CONFIG.CONF_THRESHOLD, iou=CONFIG.IOU_THRESHOLD, verbose=False,
        )
        annotated = results[0].plot()  # numpy BGR image with boxes+masks+labels drawn
        out_path = os.path.join(CONFIG.TEST_PRED_DIR, f"pred_{img_name}")
        cv2.imwrite(out_path, annotated)
        print(f"Saved: {out_path}  ({len(results[0].boxes)} detections)")

    print(f"\nAll annotated test predictions saved to: {CONFIG.TEST_PRED_DIR}")


# ==========================================================================
# STAGE 7: EXPORT MODEL
# ==========================================================================

def stage_export(weights_path=None, fmt="onnx"):
    import shutil
    from ultralytics import YOLO

    weights_path = weights_path or _default_weights_path()
    os.makedirs(CONFIG.EXPORT_DIR, exist_ok=True)
    model = YOLO(weights_path)
    exported_path = model.export(format=fmt, imgsz=CONFIG.IMG_SIZE)
    dest = os.path.join(CONFIG.EXPORT_DIR, os.path.basename(exported_path))
    shutil.copy(exported_path, dest)
    print(f"Exported model ({fmt}) -> {dest}")
    return dest


# ==========================================================================
# CLI ENTRYPOINT
# ==========================================================================

def main():
    parser = argparse.ArgumentParser(description="One-file COCO instance segmentation pipeline")
    sub = parser.add_subparsers(dest="stage", required=True)

    sub.add_parser("download", help="download dataset from Roboflow")
    sub.add_parser("analyze", help="run EDA on the COCO-format dataset")

    p_pre = sub.add_parser("preprocess", help="validate/clean the YOLO-format dataset")
    p_pre.add_argument("--fix", action="store_true", help="remove unreadable images and orphan labels")

    sub.add_parser("train", help="fine-tune YOLOv8-seg on the dataset")

    p_eval = sub.add_parser("evaluate", help="run Ultralytics val() + COCOeval")
    p_eval.add_argument("--weights", default=None)

    p_test = sub.add_parser("test", help="run inference on sample test images")
    p_test.add_argument("--weights", default=None)
    p_test.add_argument("--n", type=int, default=12)

    p_export = sub.add_parser("export", help="export trained model")
    p_export.add_argument("--weights", default=None)
    p_export.add_argument("--format", default="onnx",
                           choices=["onnx", "torchscript", "openvino", "engine", "coreml"])

    sub.add_parser("all", help="run every stage start to finish")

    args = parser.parse_args()

    if args.stage == "download":
        stage_download()
    elif args.stage == "analyze":
        stage_analyze()
    elif args.stage == "preprocess":
        stage_preprocess(fix=args.fix)
    elif args.stage == "train":
        stage_train()
    elif args.stage == "evaluate":
        stage_evaluate(weights_path=args.weights)
    elif args.stage == "test":
        stage_test(weights_path=args.weights, n_images=args.n)
    elif args.stage == "export":
        stage_export(weights_path=args.weights, fmt=args.format)
    elif args.stage == "all":
        stage_download()
        stage_analyze()
        stage_preprocess()
        best = stage_train()
        stage_evaluate(weights_path=best)
        stage_test(weights_path=best)
        stage_export(weights_path=best)


if __name__ == "__main__":
    main()

NameError: name 'os' is not defined

In [ ]:
def download():
    rf = Roboflow(api_key=config.ROBOFLOW_API_KEY)
    project = rf.workspace(config.ROBOFLOW_WORKSPACE).project(config.ROBOFLOW_PROJECT)
    version = project.version(config.ROBOFLOW_VERSION)

    print(f"Downloading '{config.ROBOFLOW_PROJECT}' v{config.ROBOFLOW_VERSION} "
          f"in YOLOv8-seg format -> {config.DATA_DIR}")
    version.download(config.ROBOFLOW_FORMAT_YOLO, location=config.DATA_DIR)

    print(f"Downloading '{config.ROBOFLOW_PROJECT}' v{config.ROBOFLOW_VERSION} "
          f"in COCO-segmentation format -> {config.COCO_DATA_DIR}")
    version.download(config.ROBOFLOW_FORMAT_COCO, location=config.COCO_DATA_DIR)

    print("\nDone. Dataset structure:")
    print(f"  YOLO format : {config.DATA_DIR}/{{train,valid,test}}/{{images,labels}}, data.yaml")
    print(f"  COCO format : {config.COCO_DATA_DIR}/{{train,valid,test}}/_annotations.coco.json")


download()